# Gerar o dataset: PIB per capita dos municípios do RN

Este notebook faz o mesmo trabalho do arquivo `gerar_dataset.py`: **baixa os dados oficiais do IBGE**, calcula o **percentil 10** e grava os arquivos em `data/`.

Cada bloco de **texto** explica o que vai acontecer. No **código**, as linhas com `#` são comentários para quem não programa (o Python não executa essas linhas).

**CSV não é necessário para começar.** Este notebook é quem **cria** os CSV e o Excel.

## Dois notebooks, duas funções

| Arquivo | Quando usar |
| --- | --- |
| **Este** (`gerar_dataset.ipynb`) | Primeiro. Coleta os dados e salva `data/`. |
| `atividade_separatrizes_pib_rn.ipynb` | Depois. Responde as perguntas da atividade. |

## O que será gravado em `data/`

| Arquivo | Conteúdo |
| --- | --- |
| `municipios_rn.csv` | 167 municípios com PIB e variáveis explicativas |
| `grupo_p10_menores_pib.csv` | Só quem está abaixo do percentil 10 |
| `municipios_rn.xlsx` | Excel com as abas da atividade |
| `resumo_separatriz.json` | P10, lista do grupo e descrição |

## Como executar

1. **Run All** / **Executar tudo**, na ordem.
2. A coleta precisa de **internet** (cerca de 1 minuto).
3. No **Colab**, clone o repositório **inteiro** (precisa da pasta `src/`). Não envie só este `.ipynb`.

```
!git clone https://github.com/AndressaLF/PIB_Munincipios_RN.git
%cd PIB_Munincipios_RN
```

Portal: [IBGE Cidades@ — RN](https://cidades.ibge.gov.br/brasil/rn/panorama). Essa página **não é uma API**; os dados saem dos serviços oficiais do IBGE.

## Mini glossário

| Palavra | Significado |
| --- | --- |
| **API** | Endereço na internet que devolve dados (JSON), não uma página bonita. |
| **JSON** | Formato de texto que o IBGE envia; o Python transforma isso em tabela. |
| **DataFrame** | Nome que o pandas dá para uma **tabela** (linhas = municípios). |
| **UF 24** | Código do IBGE para o Rio Grande do Norte. |
| **N6[N3[24]]** | “Todos os municípios (N6) que estão no estado 24”. |
| **Percentil 10** | Separatriz: cerca de 10% dos municípios ficam com PIB per capita abaixo desse valor. |

No código, tudo depois de `#` é só explicação.

## 0. Instalar ferramentas e achar a pasta do projeto

- **pandas**: tabelas
- **numpy**: percentil
- **openpyxl**: Excel

No Colab as bibliotecas são instaladas nesta célula. Depois o notebook procura a pasta `src/` (o código que fala com o IBGE).

In [ ]:
# Instala as bibliotecas. No Colab isso é necessário na primeira execução.
%pip install -q pandas numpy openpyxl matplotlib

import os
import sys
from pathlib import Path

import pandas as pd  # tabelas
import numpy as np   # percentil (usado dentro de src/pipeline.py)

pd.set_option("display.max_rows", 12)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

# True se estivermos no Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_GITHUB = "https://github.com/AndressaLF/PIB_Munincipios_RN.git"
raiz = Path.cwd()  # pasta atual

# No Colab, se src/ não estiver aqui, clona o GitHub (o código precisa já estar publicado)
if IN_COLAB and not (raiz / "src" / "pipeline.py").exists():
    destino = Path("/content/PIB_Munincipios_RN")
    if not (destino / "src" / "pipeline.py").exists():
        print("Clonando o repositório no Colab...")
        os.system(f"git clone {REPO_GITHUB} {destino}")
    if (destino / "src" / "pipeline.py").exists():
        os.chdir(destino)
        raiz = destino

# Se o notebook foi aberto numa subpasta, sobe um nível até achar src/
if not (raiz / "src" / "pipeline.py").exists() and (raiz.parent / "src" / "pipeline.py").exists():
    raiz = raiz.parent
    os.chdir(raiz)

sys.path.insert(0, str(raiz))  # permite: from src.pipeline import ...

if not (raiz / "src" / "pipeline.py").exists():
    raise FileNotFoundError(
        "Não achei a pasta src/. No Colab, publique o projeto no GitHub "
        "ou envie a pasta inteira do repositório, não só este notebook. "
        "CSV não substitui src/: este notebook é quem gera os CSV."
    )

from src.ibge_client import listar_municipios_rn
from src.pipeline import coletar_dados, resumo_separatriz, exportar

print("Pasta do projeto:", raiz)
print("Rodando no Google Colab:", "sim" if IN_COLAB else "não")
print("Código do IBGE importado com sucesso.")

## 1. Listar todos os municípios do RN

O IBGE identifica o estado pelo código **24**. Esta consulta devolve os **167 municípios**, com nome, código e recortes territoriais (mesorregião, microrregião).

Endereço usado:

```
https://servicodados.ibge.gov.br/api/v1/localidades/estados/24/municipios?orderBy=nome
```

Função no projeto: `listar_municipios_rn()` em `src/ibge_client.py`.

In [ ]:
# DataFrame = tabela. Cada linha será um município do RN.
municipios = pd.DataFrame(listar_municipios_rn())

print("Quantidade de municípios:", len(municipios))
print("Colunas:", list(municipios.columns))
print()
print("Primeiros nomes (ordem alfabética do IBGE):")
municipios.head()

## 2. O que a coleta junta em seguida

A função `coletar_dados()` (em `src/pipeline.py`) busca, para **cada município**:

| Passo | Fonte | O que entra na tabela |
| --- | --- | --- |
| PIB per capita e PIB total (2023) | Cidades@, pesquisa 38, indicadores `47001` e `46997` | Colunas `pib_per_capita` e `pib_mil_reais` |
| Valor adicionado por setor (2021) | Pesquisa 38, indicadores `47006` a `47009` | Agropecuária, indústria, serviços e administração pública |
| População, área, densidade | SIDRA tabela 4714 (Censo 2022) | `populacao_censo_2022`, `area_km2`, `densidade_demografica` |
| Alfabetização 15+ | SIDRA tabela 9543 | `taxa_alfabetizacao_15_mais_pct` |
| IDHM, escolarização, salário, ocupação | Pesquisas 10111 e 10058 | Indicadores sociais |

Recorte territorial de todos esses pedidos: **`N6[N3[24]]`** (municípios do RN).

No fim, ela também **calcula o percentil 10** e marca quem está abaixo (`abaixo_percentil_10`).

A célula seguinte **dispara todas essas consultas**. Espere cerca de um minuto.

In [ ]:
# coletar_dados() faz vários GET nas APIs, monta uma linha por município
# e já calcula o percentil 10 (coluna abaixo_percentil_10).
print("Iniciando a coleta nas APIs do IBGE (cerca de 1 minuto)...")
tabela = coletar_dados()
print()
print("Pronto. O que shape significa: (número de linhas, número de colunas).")
print("Linhas (municípios):", tabela.shape[0])
print("Colunas (variáveis):", tabela.shape[1])
tabela.head()

## 3. Conferir o percentil 10 (já calculado na coleta)

O cálculo, por dentro de `aplicar_separatriz()`, é este:

```python
percentil_10 = round(numpy.percentile(pib_per_capita, 10), 2)
abaixo_percentil_10 = pib_per_capita < percentil_10
```

- `numpy.percentile(..., 10)` ordena os valores e interpola na posição 10%.
- A coluna `abaixo_percentil_10` vale **True** só para quem tem PIB per capita **estritamente menor** que o P10 (regra da pergunta 2.2).

A célula abaixo só **lê** o que a coleta já gravou; não calcula de novo.

In [ ]:
# O P10 é gravado em todas as linhas; iloc[0] pega o valor da primeira
p10 = tabela["percentil_10_pib_per_capita"].iloc[0]
# True/False na coluna abaixo_percentil_10 já veio de coletar_dados()
grupo = tabela[tabela["abaixo_percentil_10"]].sort_values("pib_per_capita")

print("Percentil 10 (R$):", f"{p10:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print("Municípios abaixo do P10:", int(tabela["abaixo_percentil_10"].sum()))
print()
print("Grupo (menor → maior PIB per capita):")
for i, nome in enumerate(grupo["municipio"], start=1):
    print(f"  {i:2d}. {nome}")

## 4. Montar o resumo da atividade

A função `resumo_separatriz()` (em `src/pipeline.py`) faz três coisas:

1. lê o P10 já gravado na tabela;
2. compara **médias** do grupo abaixo do P10 com as dos demais municípios;
3. escreve o parágrafo da **descrição sumária** (pergunta 3 da atividade).

O resultado é um dicionário Python (pares nome → valor), não uma tabela.

In [ ]:
# resumo é um dicionário Python: pares de nome → valor (como um formulário)
resumo = resumo_separatriz(tabela)

print("Estado:", resumo["estado"])
print("Ano do PIB:", resumo["ano_pib"])
print("Total de municípios:", resumo["n_municipios"])
print("Percentil 10 (R$):", resumo["percentil_10_pib_per_capita"])
print("Municípios no grupo P10:", resumo["n_municipios_abaixo_p10"])
print()
print("Descrição sumária (texto automático a partir dos números):")
print()
print(resumo["descricao"])

## 5. Gravar CSV, Excel e JSON

`exportar()` cria a pasta `data/` (se ainda não existir) e salva:

- CSV completo e CSV só do grupo P10
- XLSX com as abas `Todos_municipios`, `Grupo_P10`, `Comparativo` e `Resumo`
- JSON com o resumo

Esses arquivos **não vão para o GitHub** (estão no `.gitignore`). Cada pessoa gera os dados na própria máquina ou no Colab.

In [ ]:
# Cria data/ e grava CSV, XLSX e JSON (esses arquivos não vão para o GitHub)
arquivos = exportar(tabela, resumo)

print("Arquivos gerados:")
for tipo, caminho in arquivos.items():
    print(f"  - {tipo}: {caminho}")
print()
print("No Colab: baixe os arquivos pela pasta à esquerda da tela.")

## 6. Olhar um pedaço da tabela final

Abaixo aparecem só algumas colunas, para caber na tela: nome, mesorregião, PIB per capita, população, peso da administração pública e da indústria, e se o município está abaixo do P10.

`head(10)` mostra as **10 primeiras linhas**. Se a coleta ordenou do menor PIB per capita para o maior, o grupo P10 aparece no topo.

In [ ]:
# Só algumas colunas, para a tabela caber na tela
colunas_principais = [
    "municipio",
    "mesorregiao",
    "pib_per_capita",
    "populacao_censo_2022",
    "participacao_adm_publica_pct",
    "participacao_industria_pct",
    "abaixo_percentil_10",
]
# head(10) = as 10 primeiras linhas (já vêm com o grupo P10 no topo, se a coleta ordenou assim)
tabela[colunas_principais].head(10)

## Pronto

O dataset da atividade foi gerado.

| Próximo passo | Arquivo |
| --- | --- |
| Responder as perguntas 1, 2.1, 2.2, 2.3 | `atividade_separatrizes_pib_rn.ipynb` (Run All) |
| Coletar de novo no terminal | `python gerar_dataset.py` |

URLs oficiais do IBGE e dicionário das colunas: `README.md`.